In [1]:
from pathlib import Path
import pandas as pd


INPUT_FILE = Path("/Users/tanglikun/Desktop/all_a_share_stock_list_20260705_130506.csv")
OUTPUT_FILE = Path("/Users/tanglikun/Desktop/A股AI产业链召回总表_第一轮.xlsx")


BASIC_AI_KEYWORDS = [
    "人工智能", "AI", "AIGC", "ChatGPT", "大模型", "生成式AI",
    "算力", "智算", "算法", "机器学习", "深度学习", "神经网络",
    "自然语言处理", "NLP", "计算机视觉", "机器视觉", "语音识别",
    "机器人", "自动驾驶", "智能驾驶", "AI芯片", "GPU",
    "AI服务器", "数据中心", "云计算"
]

OBVIOUS_NON_AI_WORDS = [
    "白酒", "啤酒", "葡萄酒", "黄酒", "饮料", "乳品", "食品",
    "调味品", "休闲食品", "肉制品", "养殖", "种植", "饲料",
    "农业", "林业", "渔业", "旅游", "酒店", "餐饮", "景区",
    "房地产", "物业", "百货", "零售", "服装", "家纺", "纺织",
    "珠宝", "化妆品", "造纸", "包装印刷", "水泥", "钢铁",
    "煤炭", "化肥", "农药", "港口", "航运", "公路", "铁路运输",
    "银行", "保险", "证券", "信托"
]

AI_LAYER_MAP = {
    "人工智能": "综合AI",
    "AI": "综合AI",
    "AIGC": "模型与应用层",
    "ChatGPT": "模型与应用层",
    "大模型": "模型与软件层",
    "生成式AI": "模型与软件层",
    "算力": "AI基础层",
    "智算": "AI基础层",
    "算法": "模型与软件层",
    "机器学习": "模型与软件层",
    "深度学习": "模型与软件层",
    "神经网络": "模型与软件层",
    "自然语言处理": "模型与软件层",
    "NLP": "模型与软件层",
    "计算机视觉": "模型与软件层",
    "机器视觉": "AI应用层",
    "语音识别": "AI应用层",
    "机器人": "AI应用层",
    "自动驾驶": "AI应用层",
    "智能驾驶": "AI应用层",
    "AI芯片": "AI基础层",
    "GPU": "AI基础层",
    "AI服务器": "AI基础层",
    "数据中心": "AI基础层",
    "云计算": "AI基础层",
}


def read_table(path):
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, dtype=str)
    return pd.read_csv(path, dtype=str, encoding="utf-8-sig")


def find_hits(text, keywords):
    text = str(text)
    return [kw for kw in keywords if kw in text]


def get_hit_text(text, hits, max_len=180):
    if not hits:
        return ""
    text = str(text).replace("\n", " ")
    for kw in hits:
        pos = text.find(kw)
        if pos != -1:
            start = max(pos - 60, 0)
            end = min(pos + 120, len(text))
            return text[start:end][:max_len]
    return ""


df = read_table(INPUT_FILE)
df = df.fillna("")

search_text = df.astype(str).agg(" ".join, axis=1)

is_ai_list = []
source_list = []
hit_keyword_list = []
hit_text_list = []
ai_layer_list = []
confidence_list = []
review_status_list = []
remark_list = []

for text in search_text:
    ai_hits = find_hits(text, BASIC_AI_KEYWORDS)
    non_ai_hits = find_hits(text, OBVIOUS_NON_AI_WORDS)

    if ai_hits:
        is_ai = "是"
        source = "第一层基础关键词召回"
        hit_keywords = "、".join(ai_hits)
        hit_text = get_hit_text(text, ai_hits)
        layers = sorted(set(AI_LAYER_MAP.get(k, "未分类") for k in ai_hits))
        ai_layer = "、".join(layers)
        confidence = "中"
        review_status = "待复核"
        remark = "当前仅基于A股原始表字段初筛，后续需用年报、主营业务、经营范围、公司简介继续验证"

    elif non_ai_hits:
        is_ai = "否"
        source = "明显非AI行业/业务排除"
        hit_keywords = "、".join(non_ai_hits)
        hit_text = get_hit_text(text, non_ai_hits)
        ai_layer = ""
        confidence = "中"
        review_status = ""
        remark = "未命中第一层AI关键词，且命中明显非AI行业/业务词；后续如有年报证据可重新纳入"

    else:
        is_ai = "不能确定"
        source = "第一轮未召回"
        hit_keywords = ""
        hit_text = ""
        ai_layer = ""
        confidence = "低"
        review_status = "进入下一轮"
        remark = "现有字段不足，需要继续用第二层关键词、公司简介、主营业务、经营范围、年报文本或概念标签判断"

    is_ai_list.append(is_ai)
    source_list.append(source)
    hit_keyword_list.append(hit_keywords)
    hit_text_list.append(hit_text)
    ai_layer_list.append(ai_layer)
    confidence_list.append(confidence)
    review_status_list.append(review_status)
    remark_list.append(remark)


df["是否在AI产业链中"] = is_ai_list
df["召回来源"] = source_list
df["命中关键词"] = hit_keyword_list
df["命中文本"] = hit_text_list
df["AI产业链环节"] = ai_layer_list
df["确定性等级"] = confidence_list
df["复核状态"] = review_status_list
df["备注"] = remark_list

df.to_excel(OUTPUT_FILE, index=False)

print("完成！")
print(f"输出文件：{OUTPUT_FILE}")
print()
print(df["是否在AI产业链中"].value_counts())

完成！
输出文件：/Users/tanglikun/Desktop/A股AI产业链召回总表_第一轮.xlsx

是否在AI产业链中
不能确定    5377
否        149
是          1
Name: count, dtype: int64
